In [ ]:
def load_projects_from_csv(csv_file: str) -> Tuple[Dict, pd.DataFrame]:
    """从CSV文件读取项目信息（支持开放时间窗）"""
    df = pd.read_csv(csv_file)
    
    project_info = {}
    
    for idx, row in df.iterrows():
        proj_id = int(row['项目ID'])
        
        info = {
            'name': row['项目名称'],
            'duration': float(row['游玩时长（分钟）']),
            'type': 'show' if row['是否演出'] == '是' else 'normal',
            'features': [
                float(row['刺激度']),
                float(row['沉浸度']),
                float(row['互动度']),
                float(row['休闲度'])
            ]
        }
        
        # 时间窗（所有项目都有）
        # 演出类：时间窗表示演出开始和最晚入场时间
        # 普通类：时间窗表示项目开放时间段
        if pd.notna(row['时间窗开始']) and pd.notna(row['时间窗结束']):
            info['time_window'] = (float(row['时间窗开始']), float(row['时间窗结束']))\n        else:\n            # 如果没有指定，默认全天开放\n            info['time_window'] = (CONFIG.PARK_OPEN_TIME, CONFIG.PARK_CLOSE_TIME)
        
        # 动态排队参数（仅普通项目）
        if info['type'] == 'normal':
            info['base_q'] = float(row['基础排队'])
            peaks = []
            if pd.notna(row['峰值1强度']):
                peaks.append((
                    float(row['峰值1强度']) * CONFIG.GMM_PEAK_INTENSITY_SCALE,
                    float(row['峰值1时间']),
                    float(row['峰值1宽度']) * CONFIG.GMM_PEAK_WIDTH_SCALE
                ))
            if pd.notna(row['峰值2强度']):
                peaks.append((
                    float(row['峰值2强度']) * CONFIG.GMM_PEAK_INTENSITY_SCALE,
                    float(row['峰值2时间']),
                    float(row['峰值2宽度']) * CONFIG.GMM_PEAK_WIDTH_SCALE
                ))
            info['peaks'] = peaks
        
        project_info[proj_id] = info
    
    return project_info, df

# 读取数据
project_info, df_projects = load_projects_from_csv('projects_data.csv')

print(f\"✓ 成功读取 {len(project_info)} 个项目\")\nprint(f\"\\n项目列表:\")\nfor proj_id, info in list(project_info.items())[:5]:\n    print(f\"  {proj_id}. {info['name']} ({info['type']})\")\nprint(f\"  ...\")\nprint(f\"\\n数据预览:\")\ndf_projects.head()

### 1.2 读取项目数据

# 迪士尼乐园路线优化系统
## 第一部分：数据读取与可视化

In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import matplotlib
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'STHeiti']
matplotlib.rcParams['axes.unicode_minus'] = False

print("✓ 导入库完成")

### 1.1 配置参数

In [ ]:
class CONFIG:
    """全局配置类 - 集中管理所有超参数"""
    
    # ========== 算法选择 ==========
    ALGORITHM = 1  # 0=遗传算法, 1=模拟退火, 2=蚁群算法
    
    # ========== 人群偏好权重矩阵 ==========
    CROWD_WEIGHTS = {
        '普通': [0.25, 0.25, 0.25, 0.25],
        '亲子': [0.10, 0.20, 0.40, 0.30],
        '情侣': [0.40, 0.40, 0.10, 0.10]
    }
    
    # ========== 日期类型排队系数 ==========
    DATE_QUEUE_MULTIPLIER = {
        '工作日': 0.7,
        '双休日': 1.0,
        '节假日': 1.5
    }
    
    # ========== 人的行走速度系数 ==========
    WALK_SPEED_MULTIPLIER = 1.0
    
    # ========== GMM动态排队模型参数 ==========
    GMM_PEAK_INTENSITY_SCALE = 1.0
    GMM_PEAK_WIDTH_SCALE = 1.0
    
    # ========== 目标函数权重参数 ==========
    WEIGHT_QUEUE_TIME = 0.1
    WEIGHT_WAIT_TIME = 0.15
    WEIGHT_WALK_TIME = 0.05
    WEIGHT_OVERTIME = 5.0
    WEIGHT_MISSED_SHOW = 1000.0
    WEIGHT_DIVERSITY = 2.0
    
    # ========== 时间约束 ==========
    PARK_OPEN_TIME = 0
    PARK_CLOSE_TIME = 1260  # 21小时 = 1260分钟
    START_TIME = 0
    
    # ========== 遗传算法参数 ==========
    GA_POPULATION_SIZE = 100
    GA_GENERATIONS = 200
    GA_CROSSOVER_RATE = 0.8
    GA_MUTATION_RATE = 0.2
    GA_ELITE_SIZE = 10
    
    # ========== 模拟退火参数 ==========
    SA_INITIAL_TEMP = 1000
    SA_COOLING_RATE = 0.995
    SA_MAX_ITERATIONS = 5000
    
    # ========== 蚁群算法参数 ==========
    ACO_ANT_COUNT = 50
    ACO_ITERATIONS = 100
    ACO_ALPHA = 1.0
    ACO_BETA = 2.0
    ACO_RHO = 0.5
    ACO_Q = 100
    
    # ========== 可视化参数 ==========
    FIGURE_DPI = 100
    FIGURE_SIZE = (12, 8)

print("✓ 配置参数加载完成")